In [1]:
import numpy as np
import pandas as pd
import spacy
import codecs, sys
import random
from collections import Counter
import pickle
from nltk.tokenize import word_tokenize
import pandas as pd
from transformers import BertModel, BertTokenizer
import torch
from allennlp.commands.elmo import ElmoEmbedder
import numpy as np
import fasttext
from sentence_transformers import SentenceTransformer
import pandas as pd
import json
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Embedding, LSTM, Bidirectional, GRU, Dense, Dropout, 
                                     GlobalMaxPooling1D, Conv1D, Flatten, Input, SimpleRNN)
from tensorflow.keras.callbacks import EarlyStopping
from sentence_transformers import SentenceTransformer
import fasttext
import fasttext.util
import logging
logging.basicConfig(level=logging.INFO)


ModuleNotFoundError: No module named 'spacy'

In [ ]:
import pandas as pd
import json

class StopwordRemover:
    def __init__(self, haseeb_json=None, chtgpt_csv=None):
        """
        Initialize the StopwordRemover class with optional stopwords sources.
        
        Args:
            haseeb_json (str): JSON string containing Haseeb Elahi's stopwords.
            chtgpt_csv (str): Path to the CSV file containing Chtgpt's stopwords.
        """
        self.haseeb_stopwords = set(self._load_haseeb_stopwords(haseeb_json)) if haseeb_json else None
        self.chtgpt_stopwords = set(self._load_chtgpt_stopwords(chtgpt_csv)) if chtgpt_csv else None

    def _load_haseeb_stopwords(self, json_string):
        """
        Load stopwords from Haseeb Elahi's JSON string.
        
        Args:
            json_string (str): JSON string containing stopwords.
        
        Returns:
            list: List of stopwords.
        """
        stop_words_data = json.loads(json_string)
        return stop_words_data.get("roman_urdu_stop_words", [])

    def _load_chtgpt_stopwords(self, csv_path):
        """
        Load stopwords from Chtgpt's CSV file.
        
        Args:
            csv_path (str): Path to the CSV file containing stopwords.
        
        Returns:
            list: List of stopwords.
        """
        df = pd.read_csv(csv_path, header=None)  # Load CSV with no column names
        return df.iloc[:, 0].tolist()  # Extract the first column as stopwords list

    def remove_stopwords(self, tokenized_list, method='haseeb'):
        """
        Remove stopwords from a tokenized list using the specified method.
        
        Args:
            tokenized_list (list): List of tokens (words).
            method (str): Method to use for stopwords removal ('haseeb' or 'chtgpt').
        
        Returns:
            list: Tokenized list without stopwords.
        """
        if method == 'haseeb':
            if not self.haseeb_stopwords:
                raise ValueError("Haseeb Elahi stopwords not loaded. Provide a valid JSON string.")
            return [word for word in tokenized_list if word.lower() not in self.haseeb_stopwords]
        
        elif method == 'chtgpt':
            if not self.chtgpt_stopwords:
                raise ValueError("Chtgpt stopwords not loaded. Provide a valid CSV path.")
            return [word for word in tokenized_list if word.lower() not in self.chtgpt_stopwords]
        
        else:
            raise ValueError(f"Unsupported method: {method}")


class Tokenizer:
    def __init__(self, dictionary_path=None):
        """
        Initialize the Tokenizer class with optional dictionary path.
        """
        self.nlp = spacy.blank('xx')  # Blank spaCy pipeline for multi-language support
        self.bert_tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')
        self.dictionary = self.load_dictionary(dictionary_path) if dictionary_path else None

    def load_dictionary(self, filepath):
        """
        Load a dictionary from a pickle file.
        """
        try:
            with open(filepath, 'rb') as f:
                return pickle.load(f)
        except Exception as e:
            print(f"Error loading dictionary: {e}")
            return None

    def tokenize(self, text, model_name='spacy'):
        """
        Tokenize the input text using the specified model.
        Supported models: 'bert', 'nltk', 'spacy', 'dictionary'.
        """
        if model_name == 'bert':
            return self._tokenize_bert(text)
        elif model_name == 'nltk':
            return self._tokenize_nltk(text)
        elif model_name == 'spacy':
            return self._tokenize_spacy(text)
        elif model_name == 'dictionary':
            return self._tokenize_dictionary(text)
        else:
            raise ValueError(f"Unsupported model: {model_name}")

    def _tokenize_bert(self, text):
        """
        Tokenize using BERT tokenizer.
        """
        return self.bert_tokenizer.tokenize(text)

    def _tokenize_nltk(self, text):
        """
        Tokenize using NLTK's word_tokenize.
        """
        return word_tokenize(text)

    def _tokenize_spacy(self, text):
        """
        Tokenize using spaCy.
        """
        doc = self.nlp(text)
        return [token.text for token in doc]

    def _tokenize_dictionary(self, text):
        """
        Tokenize using a custom dictionary.
        """
        if not self.dictionary:
            raise ValueError("Dictionary not loaded. Please provide a valid dictionary path.")
        
        # Lowercase the text
        text = text.lower()

        # Remove punctuation
        punctuation = '''!()%\n٪-;۔،:\n\/'"\,“./؟_ء'''
        for char in punctuation:
            text = text.replace(char, '')

        # Split into words
        tokens = text.split()

        # Handle bigrams using the dictionary
        bi_tokens = []
        i = 0
        while i < len(tokens):
            if i + 1 < len(tokens) and f"{tokens[i]} {tokens[i + 1]}" in self.dictionary:
                bi_tokens.append(f"{tokens[i]} {tokens[i + 1]}")
                i += 2
            else:
                bi_tokens.append(tokens[i])
                i += 1

        return bi_tokens


class EmbeddingGenerator:
    def __init__(self, bert_model_name='bert-base-uncased', elmo_options=None, elmo_weights=None, fasttext_model_path=None,sbert_model_name='sentence-transformers/all-MiniLM-L6-v2'):
        """
        Initialize the EmbeddingGenerator class with optional paths to embedding models.
        
        Args:
            bert_model_name (str): Name or path of the BERT model.
            elmo_options (str): Path to ELMo options file.
            elmo_weights (str): Path to ELMo weights file.
            fasttext_model_path (str): Path to FastText model file.
        """
        # Load BERT model
        self.bert_tokenizer = BertTokenizer.from_pretrained(bert_model_name)
        self.bert_model = BertModel.from_pretrained(bert_model_name)
        self.bert_model.eval()

        # Load ELMo model
        self.elmo = ElmoEmbedder(options_file=elmo_options, weight_file=elmo_weights) if elmo_options and elmo_weights else None

        # Load FastText model
        self.fasttext_model = fasttext.load_model(fasttext_model_path) if fasttext_model_path else None

        self.sbert_model = SentenceTransformer(sbert_model_name)


    def generate_embedding(self, sentence, model_name='bert'):
        """
        Generate embeddings for a given sentence using the specified model.
        
        Args:
            sentence (str): Input sentence.
            model_name (str): Model to use for embedding ('bert', 'elmo', 'fasttext').
        
        Returns:
            numpy.ndarray: Sentence embedding.
        """
        if model_name == 'bert':
            return self._get_bert_embedding(sentence)
        elif model_name == 'elmo':
            if not self.elmo:
                raise ValueError("ELMo model not loaded. Provide valid options and weights files.")
            return self._get_elmo_embedding(sentence)
        elif model_name == 'fasttext':
            if not self.fasttext_model:
                raise ValueError("FastText model not loaded. Provide a valid model path.")
            return self._get_fasttext_embedding(sentence)
        else:
            raise ValueError(f"Unsupported model: {model_name}")

    def _get_bert_embedding(self, sentence):
        """
        Return word-level BERT embeddings for a sentence.
        
        Returns:
            list of numpy.ndarray: Embedding for each token.
        """
        inputs = self.bert_tokenizer(sentence, return_tensors='pt', truncation=True, padding=True, return_attention_mask=True)
        with torch.no_grad():
            outputs = self.bert_model(**inputs)
        
        # Get the embeddings for all tokens (not the pooled output)
        token_embeddings = outputs.last_hidden_state.squeeze(0)  # Shape: (seq_len, hidden_size)
        return [token_embeddings[i].numpy() for i in range(token_embeddings.size(0))]

    def _get_elmo_embedding(self, sentence):
        """
        Return word-level ELMo embeddings for a sentence.
        
        Returns:
            list of numpy.ndarray: Embedding for each token.
        """
        tokens = sentence.split()
        embeddings = self.elmo.embed_sentence(tokens)  # Shape: (3 layers, seq_len, 1024)
        mean_layer = np.mean(embeddings, axis=0)  # Shape: (seq_len, 1024)
        return [mean_layer[i] for i in range(mean_layer.shape[0])]

    def _get_fasttext_embedding(self, sentence):
        """
        Return word-level FastText embeddings for a sentence.
        
        Returns:
            list of numpy.ndarray: Embedding for each token.
        """
        tokens = sentence.split()
        return [self.fasttext_model.get_word_vector(token) for token in tokens]

    def _get_fasttext_embedding(self, sentence):
        """
        Generate FastText embeddings for a given sentence.
        
        Args:
            sentence (str): Input sentence.
        
        Returns:
            numpy.ndarray: Sentence embedding.
        """
        return self.fasttext_model.get_sentence_vector(sentence)
    
    def _get_sbert_embedding(self, sentence):
        return self.sbert_model.encode(sentence)

class DeepTextClassifier:
    def __init__(self, model_type='lstm', embedding_type='word', max_len=100, embedding_dim=300):
        self.model_type = model_type.lower()
        self.embedding_type = embedding_type.lower()
        self.max_len = max_len
        self.embedding_dim = embedding_dim
        self.tokenizer = tf.keras.preprocessing.text.Tokenizer()
        self.model = None
        self.label_encoder = LabelEncoder()

        if self.embedding_type == 'sentence':
            self.sentence_model = SentenceTransformer('paraphrase-MiniLM-L6-v2')
        elif self.embedding_type == 'word':
            fasttext.util.download_model('en', if_exists='ignore')
            self.word_vectors = fasttext.load_model('cc.en.300.bin')

    def preprocess_data(self, df, text_column, label_column):
        texts = df[text_column].astype(str).tolist()
        labels = df[label_column].tolist()
        y = self.label_encoder.fit_transform(labels)

        if self.embedding_type == 'word':
            self.tokenizer.fit_on_texts(texts)
            sequences = self.tokenizer.texts_to_sequences(texts)
            X = pad_sequences(sequences, maxlen=self.max_len, padding='post')
        else:  # sentence embeddings
            X = np.array(self.sentence_model.encode(texts, show_progress_bar=True))

        return train_test_split(X, y, test_size=0.2, random_state=42)

    def build_model(self, vocab_size=None):
        model = Sequential()

        if self.embedding_type == 'word':
            model.add(Input(shape=(self.max_len,)))
            embedding_matrix = np.zeros((vocab_size, self.embedding_dim))
            for word, i in self.tokenizer.word_index.items():
                if i < vocab_size:
                    try:
                        embedding_matrix[i] = self.word_vectors.get_word_vector(word)
                    except:
                        pass

            model.add(Embedding(input_dim=vocab_size,
                                output_dim=self.embedding_dim,
                                weights=[embedding_matrix],
                                input_length=self.max_len,
                                trainable=False))

            if self.model_type == 'lstm':
                model.add(LSTM(128, return_sequences=True))
            elif self.model_type == 'bilstm':
                model.add(Bidirectional(LSTM(128, return_sequences=True)))
            elif self.model_type == 'gru':
                model.add(GRU(128, return_sequences=True))
            elif self.model_type == 'bigru':
                model.add(Bidirectional(GRU(128, return_sequences=True)))
            elif self.model_type == 'rnn':
                model.add(SimpleRNN(128, return_sequences=True))

            model.add(Dropout(0.3))
            model.add(GlobalMaxPooling1D())

        else:  # sentence embeddings
            model.add(Input(shape=(self.embedding_dim,)))

            if self.model_type == 'cnn':
                model.add(Dense(512, activation='relu'))
                model.add(Dropout(0.3))
            elif self.model_type == 'mlp':
                model.add(Dense(256, activation='relu'))
                model.add(Dropout(0.3))

        model.add(Dense(64, activation='relu'))
        model.add(Dropout(0.3))
        model.add(Dense(1, activation='sigmoid'))  # Change to softmax for multi-class

        model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
        self.model = model
        logging.info("Model built with architecture: %s", self.model_type)

    def train_and_save(self, df, text_column, label_column, output_dir='output'):
        X_train, X_test, y_train, y_test = self.preprocess_data(df, text_column, label_column)

        if self.embedding_type == 'word':
            vocab_size = min(len(self.tokenizer.word_index) + 1, 20000)
        else:
            vocab_size = None

        self.build_model(vocab_size)

        early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
        self.model.fit(X_train, y_train,
                       validation_data=(X_test, y_test),
                       epochs=10,
                       batch_size=32,
                       callbacks=[early_stopping])

        os.makedirs(output_dir, exist_ok=True)
        model_path = os.path.join(output_dir, f"{self.model_type}_{self.embedding_type}_model.h5")
        self.model.save(model_path)
        logging.info("Model trained and saved to: %s", model_path)

    def predict(self, texts):
        if self.embedding_type == 'word':
            sequences = self.tokenizer.texts_to_sequences(texts)
            padded = pad_sequences(sequences, maxlen=self.max_len, padding='post')
            return self.model.predict(padded)
        else:
            embeddings = np.array(self.sentence_model.encode(texts))
            return self.model.predict(embeddings)


# Main Function
if __name__ == "__main__":
    # Paths to resources
    input_csv = "input_data.csv"
    output_csv = "processed_data_with_embeddings.csv"
    dictionary_path = "/content/dictionary.pkl"
    haseeb_json = '''
    {
        "roman_urdu_stop_words": [
            "ai", "ayi", "hy", "hai", "main", "ki", "tha", "koi", "ko", "sy", "woh", 
            "bhi", "aur", "wo", "yeh", "rha", "hota", "ho", "ga", "ka", "le", "lye", 
            "kr", "kar", "lye", "liye", "hotay", "waisay", "gya", "gaya", "kch", "ab",
            "thy", "thay", "houn", "hain", "han", "to", "is", "hi", "jo", "kya", "thi",
            "se", "pe", "phr", "wala", "waisay", "us", "na", "ny", "hun", "rha", "raha",
            "ja", "rahay", "abi", "uski", "ne", "haan", "acha", "nai", "sent", "photo", 
            "you", "kafi", "gai", "rhy", "kuch", "jata", "aye", "ya", "dono", "hoa", 
            "aese", "de", "wohi", "jati", "jb", "krta", "lg", "rahi", "hui", "karna", 
            "krna", "gi", "hova", "yehi", "jana", "jye", "chal", "mil", "tu", "hum", "par", 
            "hay", "kis", "sb", "gy", "dain", "krny", "tou"
        ]
    }
    '''
    chtgpt_csv = "chtgpt_stopwords.csv"
    elmo_options = "elmo_options.json"
    elmo_weights = "elmo_weights.hdf5"
    fasttext_model_path = "fasttext_model.bin"

    # Initialize classes
    stopword_remover = StopwordRemover(haseeb_json=haseeb_json, chtgpt_csv=chtgpt_csv)
    tokenizer = Tokenizer(dictionary_path=dictionary_path)
    embedding_generator = EmbeddingGenerator(
        bert_model_name='bert-base-uncased',
        elmo_options=elmo_options,
        elmo_weights=elmo_weights,
        fasttext_model_path=fasttext_model_path
    )
    classifier = DeepTextClassifier(model_type='lstm', embedding_type='word', max_len=100, embedding_dim=300)

    # Read the input CSV file
    df = pd.read_csv(input_csv)

    # Ensure the required columns exist
    if 'paragraph' not in df.columns or 'category' not in df.columns:
        raise ValueError("CSV must contain 'paragraph' and 'category' columns.")

    # Process each row
    df['tokenized_words'] = ""
    df['stopword_removed_words'] = ""
    df['embeddings'] = ""

    for index, row in df.iterrows():
        paragraph = row['paragraph']
        
        # Tokenize the paragraph
        tokenized_words = tokenizer.tokenize(paragraph, model_name='spacy')
        
        # Remove stopwords
        stopword_removed_words = stopword_remover.remove_stopwords(tokenized_words, method='haseeb')
        
        # Convert stopwords-removed list back to a string
        stopword_removed_sentence = " ".join(stopword_removed_words)
        
        # Generate embeddings (using BERT by default) list of work embeddings
        embedding = embedding_generator.generate_embedding(stopword_removed_sentence, model_name='bert')

        # Optionally, you can also generate embeddings using sentence-embedding
        embedding_sentance = embedding_generator._get_sbert_embedding(stopword_removed_sentence)
        
        # Save results to the DataFrame
        df.at[index, 'tokenized_words'] = str(tokenized_words)
        df.at[index, 'stopword_removed_words'] = str(stopword_removed_words)
        df.at[index, 'embeddings'] = str(embedding.tolist())  # Convert numpy array to list for serialization
        df.at[index, 'embeddings_sentance'] = str(embedding_sentance.tolist())  # Convert numpy array to list for serialization

    #calling deep learning  method
    classifier.train_and_save(df, text_column='paragraph', label_column='category', output_dir='output')
    # Save the processed DataFrame to a new CSV file
    df.to_csv(output_csv, index=False)
    print(f"Processed data with embeddings saved to {output_csv}")

In [ ]:
import pandas as pd
import torch
from transformers import (
    BertTokenizer, BertForSequenceClassification,
    AlbertTokenizer, AlbertForSequenceClassification,
    RobertaTokenizer, RobertaForSequenceClassification,
    ElectraTokenizer, ElectraForSequenceClassification,
    DistilBertTokenizer, DistilBertForSequenceClassification,
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from datasets import Dataset

class SentimentTrainer:
    def __init__(self):
        self.models = {
            "bert": (BertTokenizer.from_pretrained("bert-base-uncased"), BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=3)),
            "albert": (AlbertTokenizer.from_pretrained("albert-base-v2"), AlbertForSequenceClassification.from_pretrained("albert-base-v2", num_labels=3)),
            "roberta": (RobertaTokenizer.from_pretrained("roberta-base"), RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=3)),
            "electra": (ElectraTokenizer.from_pretrained("google/electra-base-discriminator"), ElectraForSequenceClassification.from_pretrained("google/electra-base-discriminator", num_labels=3)),
            "distilbert": (DistilBertTokenizer.from_pretrained("distilbert-base-uncased"), DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=3)),
            "spanbert": (AutoTokenizer.from_pretrained("SpanBERT/spanbert-base-cased"), AutoModelForSequenceClassification.from_pretrained("SpanBERT/spanbert-base-cased", num_labels=3)),
            "tinybert": (AutoTokenizer.from_pretrained("prajjwal1/bert-tiny"), AutoModelForSequenceClassification.from_pretrained("prajjwal1/bert-tiny", num_labels=3)),
            "xlmr": (AutoTokenizer.from_pretrained("xlm-roberta-base"), AutoModelForSequenceClassification.from_pretrained("xlm-roberta-base", num_labels=3)),
        }

    def prepare_data(self, df):
        le = LabelEncoder()
        df['label'] = le.fit_transform(df['Category'].str.lower())
        return train_test_split(df[['Review', 'label']], test_size=0.2, random_state=42)

    def tokenize_data(self, tokenizer, dataset):
        return dataset.map(lambda x: tokenizer(x['text'], truncation=True, padding=True), batched=True)

    def train_model(self, model_name, df):
        if model_name not in self.models:
            raise ValueError(f"Model {model_name} not supported!")

        tokenizer, model = self.models[model_name]
        train_df, val_df = self.prepare_data(df)

        train_ds = Dataset.from_pandas(train_df)
        val_ds = Dataset.from_pandas(val_df)

        train_ds = self.tokenize_data(tokenizer, train_ds)
        val_ds = self.tokenize_data(tokenizer, val_ds)

        training_args = TrainingArguments(
            output_dir=f"./{model_name}_sentiment_model",
            evaluation_strategy="epoch",
            save_strategy="epoch",
            learning_rate=2e-5,
            per_device_train_batch_size=8,
            per_device_eval_batch_size=8,
            num_train_epochs=3,
            weight_decay=0.01,
            logging_dir='./logs',
            logging_steps=10,
            load_best_model_at_end=True,
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
        )

        trainer.train()
        model.save_pretrained(f"./{model_name}_sentiment_model")
        tokenizer.save_pretrained(f"./{model_name}_sentiment_model")
        print(f"Model {model_name} saved successfully.")

# Example usage:
if __name__ == "__main__":
    df = pd.read_csv("sentiment_data.csv")  # Should have columns: 'text', 'sentiment'
    model_choice = "bert"  # You can change this to "roberta", "albert", etc.

    trainer = SentimentTrainer()
    trainer.train_model(model_choice, df)
